# Multi-Agent Systems and Agentic Patterns with smolagents

## Tutorial Overview

Welcome to this comprehensive tutorial on building multi-agent systems using **Hugging Face smolagents** with an **Azure OpenAI** deployment (`gpt-5.4-mini` or `gpt-5.6-luna`) as the model backend.

### What is smolagents?

**smolagents** is a lightweight, flexible framework developed by Hugging Face for building AI agents. It provides:
- 🛠️ Simple tool creation with decorators
- 🤖 Multiple agent types (`CodeAgent`, `ToolCallingAgent`) and multi-agent hierarchies via `managed_agents`
- 🔗 Easy integration with various LLM backends
- 📦 Minimal dependencies and straightforward API

### Why Agentic Patterns Matter

Agentic patterns are architectural approaches that enable AI systems to:
- Break down complex tasks into manageable steps
- Route requests to specialized components
- Execute tasks in parallel for efficiency
- Provide feedback loops for iterative improvement
- Orchestrate multiple agents working together

In this tutorial, we'll explore **5 key agentic patterns**:
1. **Prompt Chaining** - Sequential agent workflows
2. **Routing** - Intelligent task delegation
3. **Parallelization** - Concurrent execution
4. **Orchestrator-Worker** - Manager-worker hierarchies
5. **Evaluator-Optimizer** - Feedback-driven improvement

Let's get started! 🚀

## 1. Setup & Installation

First, let's install all required dependencies.

In [1]:
# Install required packages
# smolagents[openai]: The agent framework + OpenAI-compatible model client (used for Azure OpenAI)
# python-dotenv: Loads credentials from the .env file

%pip install -q "smolagents[openai]>=1.26" python-dotenv

## 2. Azure OpenAI Configuration

This notebook uses an **Azure OpenAI** deployment through its OpenAI-compatible **v1 endpoint**.
Credentials are read from a `.env` file in this notebook's folder (or any parent folder). Copy `example.env` to `.env` and fill it in. On **Google Colab**, upload `.env` with the 📁 *Files* panel, or just enter the values when prompted:

| Variable | Description |
|----------|-------------|
| `AZURE_OPENAI_API_KEY` | Your Azure OpenAI API key |
| `AZURE_OPENAI_ENDPOINT` | Base URL, e.g. `https://<resource>.openai.azure.com/` (the full `.../openai/v1/` URL also works) |
| `AZURE_OPENAI_DEPLOYMENT` | Deployment (model) name: `gpt-5.4-mini` or `gpt-5.6-luna` |

If a value is missing, you'll be prompted for it.

**Security Best Practice**: Never hardcode API keys in your code. Keep them in `.env` (which must never be committed or shared).

In [2]:
import getpass
import os

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))

if not os.environ.get("AZURE_OPENAI_API_KEY"):
    os.environ["AZURE_OPENAI_API_KEY"] = getpass.getpass("Enter your Azure OpenAI API key: ")
if not os.environ.get("AZURE_OPENAI_ENDPOINT"):
    os.environ["AZURE_OPENAI_ENDPOINT"] = input("Enter your Azure OpenAI endpoint (https://<resource>.openai.azure.com/): ")

# Accept either the resource URL or the full v1 base URL
AZURE_OPENAI_ENDPOINT = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/").removesuffix("/openai/v1") + "/openai/v1/"
AZURE_OPENAI_DEPLOYMENT = os.environ.get("AZURE_OPENAI_DEPLOYMENT", "gpt-5.4-mini")
print(f"Using Azure OpenAI deployment: {AZURE_OPENAI_DEPLOYMENT}")

Using Azure OpenAI deployment: gpt-5.4-mini


## 3. Model Configuration with OpenAIModel

We'll create a reusable model instance using smolagents' `OpenAIModel`, pointed at our Azure OpenAI deployment through the OpenAI-compatible v1 endpoint. This model will be used across all our agentic patterns.

> **Note:** GPT-5 family models don't support the `stop` parameter; smolagents detects this automatically and omits it.

In [3]:
from smolagents import OpenAIModel

# Create a reusable model instance backed by our Azure OpenAI deployment
# This model will be used across all our agent patterns
model = OpenAIModel(
    model_id=AZURE_OPENAI_DEPLOYMENT,
    api_base=AZURE_OPENAI_ENDPOINT,
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
)

print("✓ Model configured successfully!")
print(f"Model ID: {AZURE_OPENAI_DEPLOYMENT}")
print("Backend: Azure OpenAI (OpenAI-compatible v1 endpoint)")

✓ Model configured successfully!
Model ID: gpt-5.4-mini
Backend: Azure OpenAI (OpenAI-compatible v1 endpoint)


---

# Agentic Patterns

Now let's explore the 5 key agentic patterns with practical examples!

---

## Pattern A: Prompt Chaining

### Concept
**Prompt Chaining** connects multiple agents in a sequential workflow where the output of one agent becomes the input to the next. This pattern is ideal for:
- Multi-step transformations
- Progressive refinement
- Staged processing pipelines

### Example: Text Summarization → Translation Pipeline
We'll create a chain where:
1. **Agent A** (Summarizer) condenses a long text
2. **Agent B** (Translator) translates the summary to another language

In [4]:
from smolagents import CodeAgent, ToolCallingAgent

# Create Agent A: Text Summarizer (using ToolCallingAgent for simpler text generation)
agent_summarizer = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=2
)

# Create Agent B: Translator (using ToolCallingAgent for text generation)
agent_translator = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=2
)

# Sample long text to process
long_text = """
Artificial Intelligence has revolutionized numerous industries over the past decade.
From healthcare diagnostics to autonomous vehicles, AI systems are becoming increasingly
sophisticated. Machine learning algorithms can now detect diseases from medical images
with accuracy rivaling human experts. Natural language processing has enabled chatbots
and virtual assistants that can understand and respond to human queries. Computer vision
systems power facial recognition and object detection in security systems. The field
continues to evolve rapidly with new breakthroughs in deep learning, reinforcement
learning, and neural architecture search.
"""

print("=" * 60)
print("PATTERN A: PROMPT CHAINING")
print("=" * 60)

# Step 1: Agent A summarizes the text
print("\n🔗 Step 1: Summarizing text...")
summary = agent_summarizer.run(
    f"Summarize this text in 2 sentences:\n\n{long_text}"
)
print(f"\n📝 Summary:\n{summary}")

# Step 2: Agent B translates the summary
print("\n🔗 Step 2: Translating summary to Spanish...")
translation = agent_translator.run(
    f"Translate the following English text to Spanish. Return only the Spanish translation, nothing else:\n\n{summary}"
)
print(f"\n🌐 Translation:\n{translation}")

print("\n✅ Prompt Chain Complete!")

PATTERN A: PROMPT CHAINING

🔗 Step 1: Summarizing text...


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Summarize this text in 2 sentences:                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│ Artificial Intelligence has revolutionized numerous industries over the past decade.                            │
│ From healthcare diagnostics to autonomous vehicles, AI systems are becoming increasingly                        │
│ sophisticated. Machine learning algorithms can now detect diseases from medical images                          │
│ with accuracy rivaling human experts. Natural language processing has enabled chatbots                          │
│ and virtual assistants that can understand and respond to human queries. Computer vision                        │
│ systems power facial recognition and object detection in security systems. The field                            │
│ continues to evolve rapidly with new breakthroughs in deep learning, reinforcement                              │
│ learning, and neural architecture search.                                                                       │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Artificial Intelligence has transformed many           │
│ industries, from healthcare and autonomous vehicles to security and customer support, by enabling systems that  │
│ can diagnose diseases, understand language, and recognize images. The field is advancing quickly thanks to      │
│ breakthroughs in deep learning, reinforcement learning, and neural architecture search.'}                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Artificial Intelligence has transformed many industries, from healthcare and autonomous vehicles to 
security and customer support, by enabling systems that can diagnose diseases, understand language, and recognize 
images. The field is advancing quickly thanks to breakthroughs in deep learning, reinforcement learning, and neural
architecture search.

Final answer: Artificial Intelligence has transformed many industries, from healthcare and autonomous vehicles to 
security and customer support, by enabling systems that can diagnose diseases, understand language, and recognize 
images. The field is advancing quickly thanks to breakthroughs in deep learning, reinforcement learning, and neural
architecture search.

... (5 outputs left out)

Observations: La inteligencia artificial ha transformado muchas industrias, desde la atención sanitaria y los 
vehículos autónomos hasta la seguridad y la atención al cliente, al permitir sistemas que pueden diagnosticar 
enfermedades, comprender el lenguaje y reconocer imágenes. El campo avanza rápidamente gracias a los avances en 
aprendizaje profundo, aprendizaje por refuerzo y búsqueda de arquitectura neuronal.

Final answer: La inteligencia artificial ha transformado muchas industrias, desde la atención sanitaria y los 
vehículos autónomos hasta la seguridad y la atención al cliente, al permitir sistemas que pueden diagnosticar 
enfermedades, comprender el lenguaje y reconocer imágenes. El campo avanza rápidamente gracias a los avances en 
aprendizaje profundo, aprendizaje por refuerzo y búsqueda de arquitectura neuronal.

[Step 1: Duration 1.71 seconds| Input tokens: 992 | Output tokens: 82]


🌐 Translation:
La inteligencia artificial ha transformado muchas industrias, desde la atención sanitaria y los vehículos autónomos hasta la seguridad y la atención al cliente, al permitir sistemas que pueden diagnosticar enfermedades, comprender el lenguaje y reconocer imágenes. El campo avanza rápidamente gracias a los avances en aprendizaje profundo, aprendizaje por refuerzo y búsqueda de arquitectura neuronal.

✅ Prompt Chain Complete!


---

## Pattern B: Routing

### Concept
**Routing** uses an intelligent dispatcher that analyzes incoming requests and directs them to the appropriate specialized tool or agent. This pattern enables:
- Efficient task delegation
- Specialized processing based on request type
- Modular, maintainable architectures

### Example: Query Router with Math vs. General Chat Tools
We'll create a router that decides whether to use a Math Tool or a General Chat Tool based on the query content.

In [5]:
from smolagents import tool

# Define specialized tools using the @tool decorator

@tool
def math_calculator(expression: str) -> str:
    """
    Evaluates mathematical expressions safely.

    Args:
        expression: A mathematical expression as a string (e.g., "2 + 2", "10 * 5")

    Returns:
        The result of the calculation as a string
    """
    try:
        # Safe evaluation of mathematical expressions
        result = eval(expression, {"__builtins__": {}}, {})
        return f"Result: {result}"
    except Exception as e:
        return f"Error calculating: {str(e)}"


@tool
def general_chat(query: str) -> str:
    """
    Handles general conversation and non-mathematical queries.

    Args:
        query: The user's general question or statement

    Returns:
        A friendly conversational response
    """
    return f"I received your query: '{query}'. I'm a general chat assistant ready to help with information, advice, or conversation!"


# Create Router Agent with both specialized tools
router_agent = CodeAgent(
    tools=[math_calculator, general_chat],
    model=model,
    max_steps=3
)

print("=" * 60)
print("PATTERN B: ROUTING")
print("=" * 60)

# Test queries of different types
test_queries = [
    "What is 125 multiplied by 8?",
    "Tell me about the weather today",
    "Calculate 2 ** 10",
    "What's your favorite color?"
]

for query in test_queries:
    print(f"\n📥 Query: {query}")

    # The router agent will automatically select the appropriate tool
    result = router_agent.run(query)

    print(f"📤 Response: {result}")
    print("-" * 60)

print("\n✅ Routing Complete!")

PATTERN B: ROUTING

📥 Query: What is 125 multiplied by 8?


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 125 multiplied by 8?                                                                                    │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Error in code parsing:
Your code snippet is invalid, because the regex pattern <code>(.*?)</code> was not found in it.
Here is your code snippet:
625</code>
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
<code>
# Your python code here
</code>
Make sure to provide correct code blobs.

[Step 1: Duration 0.87 seconds| Input tokens: 2,064 | Output tokens: 10]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

... (21 outputs left out)

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("I don't have personal preferences, but a popular favorite is blue.")                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: I don't have personal preferences, but a popular favorite is blue.

[Step 2: Duration 1.16 seconds| Input tokens: 4,263 | Output tokens: 67]

📤 Response: I don't have personal preferences, but a popular favorite is blue.
------------------------------------------------------------

✅ Routing Complete!


---

## Pattern C: Parallelization

### Concept
**Parallelization** executes multiple agents or tasks concurrently to improve efficiency. This pattern is valuable for:
- Independent tasks that can run simultaneously
- Reducing total execution time
- Gathering diverse perspectives on the same input

### Example: Pros & Cons Analysis
We'll run two agents in parallel—one generating pros and another generating cons for a given topic, then aggregate their results.

In [6]:
import concurrent.futures
from typing import Dict

# Create specialized agents for pros and cons analysis
# Using ToolCallingAgent for cleaner text generation
agent_pros = ToolCallingAgent(tools=[], model=model, max_steps=2)
agent_cons = ToolCallingAgent(tools=[], model=model, max_steps=2)

def generate_pros(topic: str) -> str:
    """Generate advantages/pros for a given topic"""
    prompt = f"List 3 key advantages or pros of {topic}. Be concise and specific. Provide a numbered list with brief explanations."
    return agent_pros.run(prompt)

def generate_cons(topic: str) -> str:
    """Generate disadvantages/cons for a given topic"""
    prompt = f"List 3 key disadvantages or cons of {topic}. Be concise and specific. Provide a numbered list with brief explanations."
    return agent_cons.run(prompt)

def parallel_analysis(topic: str) -> Dict[str, str]:
    """
    Run pros and cons analysis in parallel using ThreadPoolExecutor
    """
    print(f"🔄 Starting parallel analysis for: {topic}")

    # Execute both agents concurrently
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        # Submit both tasks
        future_pros = executor.submit(generate_pros, topic)
        future_cons = executor.submit(generate_cons, topic)

        # Wait for both to complete and gather results
        pros = future_pros.result()
        cons = future_cons.result()

    return {"pros": pros, "cons": cons}


print("=" * 60)
print("PATTERN C: PARALLELIZATION")
print("=" * 60)

# Test topic
topic = "Remote Work"

# Run parallel analysis
results = parallel_analysis(topic)

# Display aggregated results
print(f"\n📊 Analysis Results for '{topic}':")
print("\n✅ PROS:")
print(results["pros"])
print("\n❌ CONS:")
print(results["cons"])

print("\n✅ Parallel Analysis Complete!")

PATTERN C: PARALLELIZATION
🔄 Starting parallel analysis for: Remote Work


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ List 3 key disadvantages or cons of Remote Work. Be concise and specific. Provide a numbered list with brief    │
│ explanations.                                                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ List 3 key advantages or pros of Remote Work. Be concise and specific. Provide a numbered list with brief       │
│ explanations.                                                                                                   │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': '1. **Flexibility** — Remote work lets employees better │
│ manage their schedules and work from locations that suit them.\n2. **No commute** — It saves time and money by  │
│ eliminating daily travel to an office.\n3. **Wider talent access** — Companies can hire skilled people from     │
│ different regions without being limited by geography.'}                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (4 outputs left out)

Observations: 1. **Isolation and weaker social connection** — Remote work can reduce casual interaction, making 
employees feel isolated and less connected to their team.
2. **Communication and coordination challenges** — Without in-person cues, misunderstandings can increase and 
collaboration may be slower, especially across time zones.
3. **Blurred work-life boundaries** — Working from home can make it harder to disconnect, leading to longer hours, 
burnout, or distractions from personal responsibilities.

Final answer: 1. **Isolation and weaker social connection** — Remote work can reduce casual interaction, making 
employees feel isolated and less connected to their team.
2. **Communication and coordination challenges** — Without in-person cues, misunderstandings can increase and 
collaboration may be slower, especially across time zones.
3. **Blurred work-life boundaries** — Working from home can make it harder to disconnect, leading to longer hours, 
burnout, or distractions from personal responsibilities.

[Step 1: Duration 2.90 seconds| Input tokens: 946 | Output tokens: 108]


📊 Analysis Results for 'Remote Work':

✅ PROS:
1. **Flexibility** — Remote work lets employees better manage their schedules and work from locations that suit them.
2. **No commute** — It saves time and money by eliminating daily travel to an office.
3. **Wider talent access** — Companies can hire skilled people from different regions without being limited by geography.

❌ CONS:
1. **Isolation and weaker social connection** — Remote work can reduce casual interaction, making employees feel isolated and less connected to their team.
2. **Communication and coordination challenges** — Without in-person cues, misunderstandings can increase and collaboration may be slower, especially across time zones.
3. **Blurred work-life boundaries** — Working from home can make it harder to disconnect, leading to longer hours, burnout, or distractions from personal responsibilities.

✅ Parallel Analysis Complete!


---

## Pattern D: Orchestrator-Worker

### Concept
The **Orchestrator-Worker** pattern uses a manager agent that decomposes complex tasks into subtasks and delegates them to worker agents. This hierarchical structure enables:
- Complex task decomposition
- Coordinated multi-step workflows
- Clear separation of planning and execution

### Example: Blog Post Creation Pipeline
A Manager Agent breaks down "Write a blog post" into subtasks (outline, introduction, body, conclusion) and assigns each to a Worker Agent.

In [7]:
from typing import List
import re

def orchestrator_worker_pattern(main_task: str) -> str:
    """
    Implements the Orchestrator-Worker pattern:
    1. Manager decomposes the task into subtasks
    2. Worker executes each subtask step-by-step
    3. Results are aggregated into final output

    Note: This pattern calls the model directly (model.generate) for cleaner output without agent verbosity
    """

    print(f"🎯 Main Task: {main_task}\n")

    # Step 1: Manager decomposes the task
    print("📋 Manager: Decomposing task into subtasks...")
    decomposition_prompt = f"""Break down this task into 3 clear, actionable subtasks: "{main_task}"
Return ONLY a numbered list (1., 2., 3.) with no additional text."""

    manager_response = model.generate([{"role": "user", "content": decomposition_prompt}])

    subtasks_text = manager_response.content
    print(f"✓ Subtasks identified:\n{subtasks_text}\n")

    # Parse subtasks into list
    subtasks = [line.strip() for line in subtasks_text.split('\n')
                if line.strip() and re.match(r'^\d+\.', line.strip())]

    # Step 2: Worker executes each subtask sequentially
    print("👷 Worker: Executing subtasks...\n")
    results = []
    conversation_history = []

    for i, subtask in enumerate(subtasks, 1):
        print(f"🔨 Subtask {i}: {subtask.split('.', 1)[1].strip() if '.' in subtask else subtask}")

        # Build context-aware prompt
        if results:
            context = "\n\n".join([f"Step {j+1} output: {results[j]}" for j in range(len(results))])
            prompt = f"""Main task: {main_task}

Previous work:
{context}

Now complete: {subtask}

Build upon previous work. Provide only the output for this step."""
        else:
            prompt = f"Main task: {main_task}\n\nComplete: {subtask}\n\nProvide only the output for this step."

        # Execute subtask
        conversation_history.append({"role": "user", "content": prompt})
        worker_response = model.generate(conversation_history)

        result = worker_response.content
        conversation_history.append({"role": "assistant", "content": result})
        results.append(result)

        print(f"✓ Completed\n")

    # Step 3: Manager selects the final result (last subtask is typically the polished version)
    print("📦 Manager: Selecting final deliverable...\n")

    # For blog post writing, the last step is usually the final polished version
    # Return only the last result (the refined, publication-ready output)
    final_output = results[-1]

    return final_output


print("=" * 60)
print("PATTERN D: ORCHESTRATOR-WORKER")
print("=" * 60)
print()

# Complex task to decompose
task = "Write a short blog post about the benefits of exercise"

# Execute orchestrator-worker pattern
final_result = orchestrator_worker_pattern(task)

print("=" * 60)
print("📄 FINAL BLOG POST:")
print("=" * 60)
print(final_result)
print("\n✅ Orchestrator-Worker Pattern Complete!")

PATTERN D: ORCHESTRATOR-WORKER

🎯 Main Task: Write a short blog post about the benefits of exercise

📋 Manager: Decomposing task into subtasks...


✓ Subtasks identified:
1. Identify 3–4 key benefits of exercise to include in the post.
2. Write a short, engaging blog post with a clear introduction, body, and conclusion.
3. Edit the draft for clarity, brevity, and a friendly tone.

👷 Worker: Executing subtasks...

🔨 Subtask 1: Identify 3–4 key benefits of exercise to include in the post.


✓ Completed

🔨 Subtask 2: Write a short, engaging blog post with a clear introduction, body, and conclusion.


✓ Completed

🔨 Subtask 3: Edit the draft for clarity, brevity, and a friendly tone.


✓ Completed

📦 Manager: Selecting final deliverable...

📄 FINAL BLOG POST:
Exercise is one of the easiest ways to improve your health and feel better every day. Whether you enjoy walking, cycling, or a quick workout at home, regular movement can make a big difference.

Exercise helps keep your heart, muscles, and bones strong. It can also boost your mood, reduce stress, and give you more energy throughout the day. Many people also find that staying active helps them sleep better at night.

The good news is that exercise does not have to be intense to be helpful. Even a little movement each day can support a healthier, happier lifestyle.

In short, exercise is a simple habit with big benefits for both body and mind.

✅ Orchestrator-Worker Pattern Complete!


---

## Pattern E: Evaluator-Optimizer

### Concept
The **Evaluator-Optimizer** pattern implements a feedback loop where:
1. A Generator Agent creates an initial solution
2. An Evaluator Agent critiques the solution
3. The Generator improves based on feedback
4. This cycle repeats until quality criteria are met

This pattern enables:
- Iterative refinement
- Quality assurance through feedback
- Self-improving systems

### Example: Story Writing with Iterative Improvement
We'll create a story, have it evaluated, and then improve it based on the critique.

In [8]:
# Create Generator and Evaluator agents
# Using ToolCallingAgent: these agents only produce text, so they don't need to write Python code
agent_generator = ToolCallingAgent(tools=[], model=model, max_steps=3)
agent_evaluator = ToolCallingAgent(tools=[], model=model, max_steps=3)

def evaluator_optimizer_loop(task: str, max_iterations: int = 2) -> Dict[str, any]:
    """
    Implements the Evaluator-Optimizer pattern with feedback loop:
    1. Generator creates initial solution
    2. Evaluator provides critique
    3. Generator improves based on feedback
    4. Repeat for max_iterations
    """

    print(f"🎯 Task: {task}\n")

    # Initial generation
    print("🎨 Iteration 1: Initial Generation")
    print("-" * 60)

    current_solution = agent_generator.run(task)
    print(f"📝 Generated:\n{current_solution}\n")

    # Iterative improvement loop
    for iteration in range(max_iterations):
        print(f"\n🔍 Iteration {iteration + 2}: Evaluation & Optimization")
        print("-" * 60)

        # Evaluator critiques the current solution
        evaluation_prompt = f"""
        Evaluate this response to the task "{task}":

        {current_solution}

        Provide specific, constructive feedback on how to improve it.
        Focus on clarity, engagement, and completeness.
        """

        critique = agent_evaluator.run(evaluation_prompt)
        print(f"💬 Critique:\n{critique}\n")

        # Generator improves based on critique
        improvement_prompt = f"""
        Original task: {task}

        Previous version:
        {current_solution}

        Feedback received:
        {critique}

        Rewrite and improve the response based on this feedback.
        """

        current_solution = agent_generator.run(improvement_prompt)
        print(f"✨ Improved Version:\n{current_solution}\n")

    return {
        "final_solution": current_solution,
        "iterations": max_iterations + 1
    }


print("=" * 60)
print("PATTERN E: EVALUATOR-OPTIMIZER")
print("=" * 60)
print()

# Task for iterative improvement
writing_task = "Write a compelling 3-sentence story about a robot learning to paint"

# Execute evaluator-optimizer loop
result = evaluator_optimizer_loop(writing_task, max_iterations=2)

print("=" * 60)
print("🏆 FINAL RESULT:")
print("=" * 60)
print(result["final_solution"])
print(f"\n✅ Completed after {result['iterations']} iterations!")
print("\n✅ Evaluator-Optimizer Pattern Complete!")

PATTERN E: EVALUATOR-OPTIMIZER

🎯 Task: Write a compelling 3-sentence story about a robot learning to paint

🎨 Iteration 1: Initial Generation
------------------------------------------------------------


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Write a compelling 3-sentence story about a robot learning to paint                                             │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.4-mini ────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'At dawn, the small workshop hummed as the robot lifted │
│ a brush in its steel hand, staring at a blank canvas like it was a new sky waiting to be named. It began by     │
│ copying the colors around it—the copper of the kettle, the blue of the window, the gold of the sun on the       │
│ floor—but soon it learned that the most beautiful paintings were the ones that made even silence feel alive.    │
│ When it finally stepped back from its finished canvas, the robot didn’t just see a picture; it saw, for the     │
│ first time, a reflection of its own imagination.'}                                                              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: At dawn, the small workshop hummed as the robot lifted a brush in its steel hand, staring at a blank 
canvas like it was a new sky waiting to be named. It began by copying the colors around it—the copper of the 
kettle, the blue of the window, the gold of the sun on the floor—but soon it learned that the most beautiful 
paintings were the ones that made even silence feel alive. When it finally stepped back from its finished canvas, 
the robot didn’t just see a picture; it saw, for the first time, a reflection of its own imagination.

Final answer: At dawn, the small workshop hummed as the robot lifted a brush in its steel hand, staring at a blank 
canvas like it was a new sky waiting to be named. It began by copying the colors around it—the copper of the 
kettle, the blue of the window, the gold of the sun on the floor—but soon it learned that the most beautiful 
paintings were the ones that made even silence feel alive. When it finally stepped back from its finished canvas, 
the robot didn’t just see a picture; it saw, for the first time, a reflection of its own imagination.

... (26 outputs left out)

Observations: At dawn, the robot stood before a blank canvas and learned its first lesson: to paint, it had to slow
down, steady its hand, and let the brush touch the surface as gently as a heartbeat. It made mistakes at 
first—colors muddied, lines wavered, and shapes collapsed—but each correction taught it something new about light, 
shadow, and the patient art of trying again. By sunset, the finished painting held the window, the kettle, and a 
bird in flight, and the robot understood that it had not only made a picture, but discovered a way to turn its 
careful efforts into something that felt alive.

Final answer: At dawn, the robot stood before a blank canvas and learned its first lesson: to paint, it had to slow
down, steady its hand, and let the brush touch the surface as gently as a heartbeat. It made mistakes at 
first—colors muddied, lines wavered, and shapes collapsed—but each correction taught it something new about light, 
shadow, and the patient art of trying again. By sunset, the finished painting held the window, the kettle, and a 
bird in flight, and the robot understood that it had not only made a picture, but discovered a way to turn its 
careful efforts into something that felt alive.

[Step 1: Duration 2.06 seconds| Input tokens: 1,245 | Output tokens: 143]

✨ Improved Version:
At dawn, the robot stood before a blank canvas and learned its first lesson: to paint, it had to slow down, steady its hand, and let the brush touch the surface as gently as a heartbeat. It made mistakes at first—colors muddied, lines wavered, and shapes collapsed—but each correction taught it something new about light, shadow, and the patient art of trying again. By sunset, the finished painting held the window, the kettle, and a bird in flight, and the robot understood that it had not only made a picture, but discovered a way to turn its careful efforts into something that felt alive.

🏆 FINAL RESULT:
At dawn, the robot stood before a blank canvas and learned its first lesson: to paint, it had to slow down, steady its hand, and let the brush touch the surface as gently as a heartbeat. It made mistakes at first—colors muddied, lines wavered, and shapes collapsed—but each correction taught it something new about light, shadow, and the patient art of trying again. By

---

# Conclusion

## Summary of Agentic Patterns

Congratulations! 🎉 You've explored 5 powerful agentic patterns with smolagents. Here's a quick reference guide:

| Pattern | When to Use | Key Benefits |
|---------|-------------|--------------|
| **Prompt Chaining** | Sequential transformations, multi-stage processing | Simple, predictable flow; easy to debug |
| **Routing** | Tasks requiring different expertise/tools | Efficient delegation; modular architecture |
| **Parallelization** | Independent tasks, diverse perspectives | Reduced latency; concurrent processing |
| **Orchestrator-Worker** | Complex tasks needing decomposition | Clear structure; manageable subtasks |
| **Evaluator-Optimizer** | Quality-critical outputs, iterative refinement | Self-improvement; quality assurance |

## Choosing the Right Pattern

- **Use Prompt Chaining** when you have a clear pipeline with dependent steps
- **Use Routing** when different request types need specialized handling  
- **Use Parallelization** when tasks are independent and time is critical
- **Use Orchestrator-Worker** when complexity requires hierarchical planning
- **Use Evaluator-Optimizer** when quality improvement through iteration is valuable

## Next Steps

1. **Combine Patterns**: Mix patterns for sophisticated workflows (e.g., Routing + Orchestrator-Worker)
2. **Add Custom Tools**: Create domain-specific tools with the `@tool` decorator
3. **Experiment with Models**: Switch `AZURE_OPENAI_DEPLOYMENT` between `gpt-5.4-mini` and `gpt-5.6-luna`, or try other smolagents backends (`LiteLLMModel`, `InferenceClientModel`, ...)
4. **Handle Errors**: Add robust error handling and retry logic
5. **Monitor Performance**: Track token usage and latency

## Resources

- 📚 [smolagents Documentation](https://huggingface.co/docs/smolagents)
- ☁️ [Azure OpenAI v1 API Documentation](https://learn.microsoft.com/azure/foundry/openai/api-version-lifecycle)
- 🔗 [smolagents Models Reference](https://huggingface.co/docs/smolagents/reference/models)

---

**Happy Building! 🚀**